# ________________________________

## Imports and initial setup


In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.embeddings.bedrock import BedrockEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings.ollama import OllamaEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.llms.ollama import Ollama
from langchain_core.documents import Document
import warnings
warnings.filterwarnings("ignore")
import argparse
import shutil
import os

### Setting up paths and the get embedding function

Setting `CHROMA_PATH`, `DATA_PATH`, and defining the `get_embedding_function`.

In [2]:
CHROMA_PATH = "chroma"
DATA_PATH = "my_pdfs"
model_path = r"D:\AI\MODELS\all-MiniLM-L6-v2" 
def get_embedding_function():
    embeddings = HuggingFaceEmbeddings(model_name=model_path)
    return embeddings

### Document splitting and preparation functions (chunk_size=700)

In [19]:
def load_documents():
    document_loader = PyPDFDirectoryLoader(DATA_PATH)
    return document_loader.load()

def split_documents(documents: list[Document]):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=80,
        length_function=len,
        is_separator_regex=False,
    )
    return text_splitter.split_documents(documents)

def calculate_chunk_ids(chunks):
    last_page_id = None
    current_chunk_index = 0
    for chunk in chunks:
        source = chunk.metadata.get("source")
        page = chunk.metadata.get("page")
        current_page_id = f"{source}:{page}"
        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0
        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
        chunk.metadata["id"] = chunk_id
    return chunks

def add_to_chroma(chunks: list[Document]):
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=get_embedding_function())
    chunks_with_ids = calculate_chunk_ids(chunks)
    existing_items = db.get(include=[])
    existing_ids = set(existing_items["ids"])
    print(f"Number of existing documents in DB: {len(existing_ids)}")

    new_chunks = [chunk for chunk in chunks_with_ids if chunk.metadata["id"] not in existing_ids]

    if len(new_chunks):
        print(f"👉 Adding new documents: {len(new_chunks)}")
        new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
        db.add_documents(new_chunks, ids=new_chunk_ids)
        db.persist()
    else:
        print("✅ No new documents to add")

def clear_database():
    if os.path.exists(CHROMA_PATH):
        shutil.rmtree(CHROMA_PATH)

### Conversation Template and Query Function

Definition of the `PROMPT_TEMPLATE` template and the `query_rag` function responsible for the query.

In [21]:
PROMPT_TEMPLATE = """
You are a QA assistant.

Answer ONLY from the provided context.

If multiple passages mention similar events,
always answer the one that exactly matches the question.

If the answer does not exist, reply:
"I don't know."
---
Context: {context}
---
Question: {question}
"""

def query_rag(query_text: str):
    embedding_function = get_embedding_function()
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)

    results = db.similarity_search_with_score(query_text, k=5)
    context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
    
    prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
    prompt = prompt_template.format(context=context_text, question=query_text)

    model = Ollama(model="gemma4:e2b")
    #model = Ollama(model="deepseek-r1:8b")
    response_text = model.invoke(prompt)    

    sources = [doc.metadata.get("id", None) for doc, _score in results]
    print(f"Response: {response_text}\nSources: {sources}")
    return "Don"

In [23]:
# clear_database() 

documents = load_documents()
chunks = split_documents(documents)
add_to_chroma(chunks)

#my_question = "How much total money does a player start with in Monopoly?"
#query_rag(my_question)

Number of existing documents in DB: 100
👉 Adding new documents: 156


## Example query: White Rabbit


In [27]:
query_rag("What was the White Rabbit wearing and carrying when Alice first saw him?")

Response: When Alice first saw the White Rabbit, she realized he had a waistcoat-pocket and a watch to take out of it. Later, when he returned, he was splendidly dressed with a pair of white kid gloves in one hand and a large fan in the other.
Sources: ['my_pdfs\\alice.pdf:2:0', 'my_pdfs\\alice.pdf:46:1', 'my_pdfs\\alice.pdf:12:0', 'my_pdfs\\alice.pdf:5:3', 'my_pdfs\\alice.pdf:31:1']


'Don'

### Bottle label inquiry

In [25]:
query_rag("What did the label on the first bottle Alice found say?")

Response: “DRINK ME,” beautifully printed on it in large letters.
Sources: ['my_pdfs\\alice.pdf:3:6', 'my_pdfs\\alice.pdf:12:1', 'my_pdfs\\alice.pdf:3:2', 'my_pdfs\\alice.pdf:4:0', 'my_pdfs\\alice.pdf:18:3']


'Don'

### Inquire about Dodo Race prizes


In [29]:
query_rag("fter the Caucus-race, what did Alice give to the other animals as prizes, and what did she receive as her own prize?")

Response: Alice handed round as prizes a box of comfits, which had exactly one a-piece, all round. She received only a thimble as her own prize.
Sources: ['my_pdfs\\alice.pdf:9:2', 'my_pdfs\\alice.pdf:9:0', 'my_pdfs\\alice.pdf:1:0', 'my_pdfs\\alice.pdf:8:0', 'my_pdfs\\alice.pdf:10:3']


'Don'

### Test Queries List

A list of `test_queries` for testing query functions.

In [31]:
test_queries = [
    "What was the jar Alice took down from one of the shelves labelled, and what was inside it?",
    "How many miles down did Alice think she had fallen when she estimated getting near the centre of the earth?",
    "How high was the little door that Alice found behind the low curtain?",
    "What mixed flavours did the liquid in the first bottle have according to Alice?",
    "How high did Alice grow after eating the cake, and how deep was the pool of tears she wept?",
    "According to Alice's confused geography, what is London the capital of, and what is Paris the capital of?",
    "What was the first sentence Alice said to the Mouse in French, and why?",
    "What argument did Alice have with the Lory on the bank, and how did the Lory respond?",
    "What historical story did the Mouse recite because it was the 'driest thing' it knew?",
    "How did the Dodo mark out the course for the Caucus-race, and how did the running start and end?"
]

for i, query in enumerate(test_queries, 1):
    print(f"--- Testing Query {i} ---")
    print(f"Question: {query}")
    
    response = query_rag(query) 
    
    print(f"Response: {response}\n")

--- Testing Query 1 ---
Question: What was the jar Alice took down from one of the shelves labelled, and what was inside it?
Response: The jar was labelled “ORANGE MARMALADE”, but it was empty.
Sources: ['my_pdfs\\alice.pdf:2:3', 'my_pdfs\\alice.pdf:2:1', 'my_pdfs\\alice.pdf:29:1', 'my_pdfs\\alice.pdf:4:1', 'my_pdfs\\alice.pdf:3:3']
Response: Don

--- Testing Query 2 ---
Question: How many miles down did Alice think she had fallen when she estimated getting near the centre of the earth?
Response: Alice thought she had fallen four thousand miles down when she estimated getting near the centre of the earth.
Sources: ['my_pdfs\\alice.pdf:2:4', 'my_pdfs\\alice.pdf:2:5', 'my_pdfs\\alice.pdf:2:1', 'my_pdfs\\alice.pdf:2:2', 'my_pdfs\\alice.pdf:19:5']
Response: Don

--- Testing Query 3 ---
Question: How high was the little door that Alice found behind the low curtain?
Response: The little door was about fifteen inches high.
Sources: ['my_pdfs\\alice.pdf:3:4', 'my_pdfs\\alice.pdf:3:1', 'my_pdfs

The developed RAG system underwent a rigorous human evaluation that demonstrated exceptional efficiency and high stability in retrieving complex details and generating factually accurate responses without any hallucination. The system proved its capability to detect leading questions and correct their premises while strictly providing precise source attributions. The only identified limitation was a minor bottleneck in retrieving distant semantic relations due to the constraints of the lightweight embedding model, which will be addressed in the future by integrating re-ranking techniques.